# Native Promoter Pipeline

從 KEGG BRITE 檔與 Data S1 Excel 抓取 **12 株**菌的 native promoter 序列，依分層邏輯挑選後輸出新 Excel。

## 挑選邏輯（每株獨立，固定 `SEED=42`，per-TSS 計數）

每株設定一個上限 `N`，依下列三層**依序**填滿，每層內隨機抽取：

1. **Pool A — 四類功能**（TL/NM/AM/SR）：優先；若 `|A| ≥ N` 則從 A 隨機抽 N，否則 A 全取。
2. **Pool B — 其他有 KEGG 註解**的基因：A 不足 N 時，從 B 隨機補足。
3. **Pool C — 無 KEGG 註解**的啟動子：A+B 仍不足 N 時，最後從 C 隨機補足。

> 序列為空的列會被排除。`Pool` 欄記錄每條來自哪一層。可用數不足 N 的物種會略低於上限，`Summary` sheet 的 `Short` 欄會記缺口。

**輸入**
- `Data_S1_20250826.xlsx`：每株一個 sheet（acronym）
- 12 個 `<org>00001.keg` KEGG BRITE htext 檔

**輸出**
- `data/native/native_promoters_12species.xlsx`：每株一個 sheet + Summary sheet
  （**不是殘留檔**，最後一格會再讀回它產生 `outputs/02_native_promoters.csv`）
- `data/native/kegg_4cat_counts.csv`：各 KEG 檔的四類 gene count
- `outputs/02_native_promoters.csv`：交給 06 的標準化表

## 物種清單與 N 上限

| Sheet | 物種 | KEGG | N |
|-------|------|------|---|
| Sy.el | Synechococcus elongatus | syu | 750 |
| My.sm | Mycolicibacterium smegmatis | msm | 750 |
| Ba.su | Bacillus subtilis | bsu | 750 |
| Ps.ae | Pseudomonas aeruginosa | pau | 750 |
| Es.co | Escherichia coli | eco | 750 |
| My.tu | Mycobacterium tuberculosis (H37Rv) | mtu | 300 |
| St.au | Staphylococcus aureus (MW2) | sam | 300 |
| Ba.th | Bacteroides thetaiotaomicron | bth | 300 |
| He.py | Helicobacter pylori (26695) | hpy | 300 |
| Ag.tu | Agrobacterium tumefaciens (C58) | atu | 300 |
| Bu.ce | Burkholderia cenocepacia (J2315) | bcj | 300 |
| Sa.ty | Salmonella enterica Typhimurium (SL1344) | sey | 300 |

> N 的實際定義在下面的 `SPECIES_CONFIG`，這張表只是說明；改參數請改程式，不要只改這裡。

## 功能分類（KEGG B 層 5 位數編號）
| Code | Label | 意義 |
|------|-------|------|
| 09122 | TL | Translation |
| 09104 | NM | Nucleotide metabolism |
| 09105 | AM | Amino acid metabolism |
| 09132 | SR | Signal transduction |


In [4]:
# === Path bootstrap (shared by 01-07) ===
# Locates MS2_Data_PyTorch/scripts/library_release by walking up from the cwd,
# then imports _paths, which sets every other path absolutely and puts
# MS2_Data_PyTorch/scripts on sys.path. Safe to run from any working directory.
import sys
from pathlib import Path

for _c in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    _rel = _c / "MS2_Data_PyTorch" / "scripts" / "library_release"
    if (_rel / "_paths.py").exists():
        if str(_rel) not in sys.path:
            sys.path.insert(0, str(_rel))
        break
else:
    raise RuntimeError(f"library_release not found from {Path.cwd()}")

from _paths import *  # noqa: F401,F403

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_DIR     :", DATA_DIR)
print("RELEASE_OUT  :", RELEASE_OUT)


PROJECT_ROOT : C:\project\Whole-model
DATA_DIR     : C:\project\Whole-model\MS2_Data_PyTorch\scripts\library_release\data
RELEASE_OUT  : C:\project\Whole-model\MS2_Data_PyTorch\scripts\library_release\outputs


In [5]:
import pandas as pd
import re
import random
from collections import defaultdict

EXCEL_FILE = require(NATIVE_DIR / "Data_S1_20250826.xlsx", "Data S1")
OUTPUT_FILE = NATIVE_DIR / "native_promoters_12species.xlsx"

SEED = 42  # 固定隨機種子 -> 結果可重現

# sheet acronym -> (KEGG BRITE 檔, N 上限)
SPECIES_CONFIG = {
    "Sy.el": ("syu00001.keg", 750),  # Synechococcus elongatus
    "My.sm": ("msm00001.keg", 750),  # Mycolicibacterium smegmatis
    "Ba.su": ("bsu00001.keg", 750),  # Bacillus subtilis
    "Ps.ae": ("pau00001.keg", 750),  # Pseudomonas aeruginosa
    "Es.co": ("eco00001.keg", 750),  # Escherichia coli
    "My.tu": ("mtu00001.keg",  300),  # Mycobacterium tuberculosis (H37Rv)
    "St.au": ("sam00001.keg",  300),  # Staphylococcus aureus (MW2)
    "Ba.th": ("bth00001.keg",  300),  # Bacteroides thetaiotaomicron
    "He.py": ("hpy00001.keg",  300),  # Helicobacter pylori (26695)
    "Ag.tu": ("atu00001.keg",  300),  # Agrobacterium tumefaciens (C58)
    "Bu.ce": ("bcj00001.keg",  300),  # Burkholderia cenocepacia (J2315)
    "Sa.ty": ("sey00001.keg",  300),  # Salmonella enterica Typhimurium (SL1344)
}

SPECIES_LIST = list(SPECIES_CONFIG.keys())
KEGG_FILES   = {sp: require(NATIVE_DIR / cfg[0], f'KEGG BRITE {sp}') for sp, cfg in SPECIES_CONFIG.items()}
N_TARGETS    = {sp: cfg[1] for sp, cfg in SPECIES_CONFIG.items()}

CATEGORY_MAP = {
    "09122": "TL",
    "09104": "NM",
    "09105": "AM",
    "09132": "SR",
}

CATEGORIES = ["TL", "NM", "AM", "SR"]

# 修改這行可調整 promoter 序列範圍：
#   核心區：["Minus35", "Spacer", "Minus10"]
#   含 ITR：PROMOTER_COLS + ["ITR"]
PROMOTER_COLS = ["UP", "Minus35", "Spacer", "Minus10", "Dis", "Start"]

OUTPUT_COLS = [
    "Species", "Locus_tag", "Locus_tag (old)", "Gene_name",
    "Pool", "Category", "multi_category",
    "TSS", "Direction",
    "UP", "Minus35", "Spacer", "Minus10", "Dis", "Start", "ITR",
    "Promoter_seq",
]

## Step 1 — 解析 KEGG BRITE 檔

KEGG htext 為縮排階層格式，行首字母代表層級：

```
A  09100 Metabolism
B    09104 Nucleotide metabolism
C      00230 Purine metabolism
D        b0001 thrL; ...<tab>K00001 ...
```

解析策略：
1. 遇到 **B 行** → 用 regex 抓 5 位數編號；對照 `CATEGORY_MAP` 決定是否為四類之一（`cur_label`），並記住目前確實在某個 B 分類底下（`in_b`）。
2. 遇到 **D 行** 且在 B 底下 → 取第一個 token 作 gene_id：
   - 一律加入 **`annotated`**（有 KEGG 註解的基因全集，供 Pool B 判定）。
   - 若 `cur_label` 屬四類 → 另記入 **`gene_4cat[gene_id]`**（供 Pool A 判定）。
3. 同一 gene_id 出現多次/多類時用 set 去重。

回傳兩個物件：`gene_4cat`（四類分類）與 `annotated`（全部有註解基因）。每株印出兩者大小供確認。

In [6]:
def parse_kegg_brite(filepath, category_map):
    """Parse KEGG BRITE htext.

    Returns:
        gene_4cat: {gene_id: set(四類 labels)}   只含 TL/NM/AM/SR
        annotated: set(gene_id)                  出現在任何 B 分類下的基因
    """
    gene_4cat = defaultdict(set)
    annotated = set()
    in_b      = False   # 目前是否在某個 B 分類底下
    cur_label = None    # 目前 B 區塊若為四類則為其 label，否則 None

    with open(filepath, "r", encoding="utf-8") as fh:
        for raw_line in fh:
            line = raw_line.rstrip("\n")
            if not line:
                continue
            first = line[0]

            if first == "B":
                m = re.search(r"\b(\d{5})\b", line)
                in_b = m is not None
                cur_label = category_map.get(m.group(1)) if m else None

            elif first == "D" and in_b:
                rest = line[1:].strip()
                gene_id = rest.split()[0] if rest.split() else None
                if gene_id:
                    annotated.add(gene_id)
                    if cur_label:
                        gene_4cat[gene_id].add(cur_label)

    return dict(gene_4cat), annotated


print("Parsing KEGG BRITE files...")
kegg_4cat  = {}
kegg_annot = {}
for sp in SPECIES_LIST:
    g4, ann = parse_kegg_brite(KEGG_FILES[sp], CATEGORY_MAP)
    kegg_4cat[sp]  = g4
    kegg_annot[sp] = ann
    per_cat = {c: sum(1 for cats in g4.values() if c in cats) for c in CATEGORIES}
    print(f"  {sp}: annotated={len(ann):5}  4cat_genes={len(g4):4} | "
          + " | ".join(f"{c}={per_cat[c]}" for c in CATEGORIES))

Parsing KEGG BRITE files...
  Sy.el: annotated= 1436  4cat_genes= 340 | TL=128 | NM=56 | AM=125 | SR=43
  My.sm: annotated= 2953  4cat_genes= 638 | TL=146 | NM=100 | AM=328 | SR=100
  Ba.su: annotated= 2575  4cat_genes= 608 | TL=202 | NM=87 | AM=208 | SR=131
  Ps.ae: annotated= 3512  4cat_genes= 814 | TL=156 | NM=94 | AM=328 | SR=265
  Es.co: annotated= 3392  4cat_genes= 701 | TL=191 | NM=122 | AM=220 | SR=188
  My.tu: annotated= 2053  4cat_genes= 445 | TL=132 | NM=71 | AM=203 | SR=61
  St.au: annotated= 1667  4cat_genes= 432 | TL=161 | NM=65 | AM=144 | SR=78
  Ba.th: annotated= 1803  4cat_genes= 443 | TL=161 | NM=69 | AM=153 | SR=75
  He.py: annotated= 1009  4cat_genes= 264 | TL=123 | NM=41 | AM=76 | SR=34
  Ag.tu: annotated= 2957  4cat_genes= 633 | TL=148 | NM=102 | AM=251 | SR=154
  Bu.ce: annotated= 3840  4cat_genes= 842 | TL=182 | NM=105 | AM=380 | SR=199
  Sa.ty: annotated= 3411  4cat_genes= 708 | TL=191 | NM=104 | AM=213 | SR=215


## KEGG 4-category counts CSV

輸出每個 KEG 檔案中 TL/NM/AM/SR 四個功能的 gene count。

In [7]:
CATEGORY_INFO = {
    "09122": ("TL", "Translation"),
    "09104": ("NM", "Nucleotide metabolism"),
    "09105": ("AM", "Amino acid metabolism"),
    "09132": ("SR", "Signal transduction"),
}

from pathlib import Path

def count_keg_categories(keg_file, category_info):
    entry_counts = {code: 0 for code in category_info}
    genes_by_code = {code: set() for code in category_info}
    current_code = None

    with open(keg_file, "r", encoding="utf-8") as fh:
        for raw_line in fh:
            line = raw_line.rstrip("\n")
            if not line:
                continue

            first = line[0]
            if first == "B":
                m = re.search(r"\b(\d{5})\b", line)
                current_code = m.group(1) if m and m.group(1) in category_info else None
            elif first == "D" and current_code:
                rest = line[1:].strip()
                gene_id = rest.split()[0] if rest.split() else None
                entry_counts[current_code] += 1
                if gene_id:
                    genes_by_code[current_code].add(gene_id)

    return entry_counts, genes_by_code

count_rows = []
for keg_file in sorted(NATIVE_DIR.rglob("*.keg")):
    entry_counts, genes_by_code = count_keg_categories(keg_file, CATEGORY_INFO)
    for code, (label, function) in CATEGORY_INFO.items():
        count_rows.append({
            "KEG_file": str(keg_file),
            "Code": code,
            "Label": label,
            "Function": function,
            "Entry_count": entry_counts[code],
            "Unique_gene_count": len(genes_by_code[code]),
        })

kegg_4cat_counts_df = pd.DataFrame(count_rows)
kegg_4cat_counts_df.to_csv(NATIVE_DIR / "kegg_4cat_counts.csv", index=False, encoding="utf-8-sig")
print("Wrote kegg_4cat_counts.csv")
display(kegg_4cat_counts_df)

Wrote kegg_4cat_counts.csv


,KEG_file,Code,Label,Function,Entry_count,Unique_gene_count
0,C:\project\Whole-model\MS2_Data_PyTorch\script...,09122,TL,Translation,148,148
1,C:\project\Whole-model\MS2_Data_PyTorch\script...,09104,NM,Nucleotide metabolism,112,102
2,C:\project\Whole-model\MS2_Data_PyTorch\script...,09105,AM,Amino acid metabolism,353,251
3,C:\project\Whole-model\MS2_Data_PyTorch\script...,09132,SR,Signal transduction,154,154
4,C:\project\Whole-model\MS2_Data_PyTorch\script...,09122,TL,Translation,182,182
5,C:\project\Whole-model\MS2_Data_PyTorch\script...,09104,NM,Nucleotide metabolism,113,105
6,C:\project\Whole-model\MS2_Data_PyTorch\script...,09105,AM,Amino acid metabolism,517,380
7,C:\project\Whole-model\MS2_Data_PyTorch\script...,09132,SR,Signal transduction,199,199
8,C:\project\Whole-model\MS2_Data_PyTorch\script...,09122,TL,Translation,202,202
9,C:\project\Whole-model\MS2_Data_PyTorch\script...,09104,NM,Nucleotide metabolism,95,87


## Step 2 — 讀取 Data S1 Excel

- `dtype=str`：所有欄位當字串讀入，避免 b 編號（如 `b0001`）被 pandas 誤判為數值而截去前導字母。
- 輸出各 sheet 的列數與欄位名稱，方便確認 header 對齊。

In [8]:
print("Reading Excel sheets...")
sheets = {}
for sp in SPECIES_LIST:
    sheets[sp] = pd.read_excel(EXCEL_FILE, sheet_name=sp, dtype=str, header=0)
    print(f"  {sp}: {len(sheets[sp])} rows, cols={list(sheets[sp].columns)}")

Reading Excel sheets...
  Sy.el: 2429 rows, cols=['TSS', 'Direction', 'Gene_type', 'Gene_start', 'Gene_end', 'Gene_name', 'Locus_tag', 'Locus_tag (old)', "Minus10_3'", 'UP', 'Minus35', 'Spacer', 'Minus10', 'Dis', 'Start', 'ITR']
  My.sm: 3043 rows, cols=['TSS', 'Direction', 'Gene_type', 'Gene_start', 'Gene_end', 'Gene_name', 'Locus_tag', 'Locus_tag (old)', "Minus10_3'", 'UP', 'Minus35', 'Spacer', 'Minus10', 'Dis', 'Start', 'ITR']
  Ba.su: 600 rows, cols=['TSS', 'Direction', 'Gene_type', 'Gene_start', 'Gene_end', 'Gene_name', 'Locus_tag', 'Locus_tag (old)', "Minus10_3'", 'UP', 'Minus35', 'Spacer', 'Minus10', 'Dis', 'Start', 'ITR']
  Ps.ae: 2117 rows, cols=['TSS', 'Direction', 'Gene_type', 'Gene_start', 'Gene_end', 'Gene_name', 'Locus_tag', 'Locus_tag (old)', "Minus10_3'", 'UP', 'Minus35', 'Spacer', 'Minus10', 'Dis', 'Start', 'ITR']
  Es.co: 1865 rows, cols=['TSS', 'Direction', 'Gene_type', 'Gene_start', 'Gene_end', 'Gene_name', 'Locus_tag', 'Locus_tag (old)', "Minus10_3'", 'UP', 'Minus3

## Step 3 — 分池、分層挑選、組 Promoter_seq

### Join key 選擇

每株自動比較 `Locus_tag` 與 `Locus_tag (old)` 兩欄對 `annotated` 的命中數，選命中高者。
locus 欄可能一格含多個 id（如 `AGR_C_9; Atu0006`、`BT0037; BT_0037`），以 `;`/`,` 切成 token，**任一 token 命中即算該列命中**。程式印出兩欄命中數供核對。

### 分池（per-TSS，每列一條）

對每列以 join 欄的 token 對照 KEGG：
- **Pool A `4cat`**：任一 token 屬 TL/NM/AM/SR（`Category` 記四類 label）。
- **Pool B `other_annotated`**：有 KEGG 註解但不屬四類。
- **Pool C `unannotated`**：完全無 KEGG 註解。

序列為空（`Promoter_seq == ""`）的列直接排除。

### 分層挑選（每株固定 `SEED=42`，可重現）

依序 A → B → C 填到 N，每層內以 `random.Random(SEED).sample` 隨機抽：
- `|A| ≥ N` → 從 A 抽 N。
- 否則 A 全取，從 B 抽 `N−|A|`；仍不足再從 C 抽。
- 三池總和 < N（如 Ag.tu）→ 全取，`Short` 欄記缺口。

### Promoter_seq 組合

串接 `PROMOTER_COLS`（UP + Minus35 + Spacer + Minus10 + Dis + Start），跳過空值，轉大寫。
**UP 只取末尾 19 bp**；其餘全取。同基因多類保留單列，`Category` 逗號分隔、`multi_category=True`。

In [9]:
def split_ids(val):
    """locus 欄可能一格含多個 id（; 或 , 分隔）-> token list。"""
    if pd.isna(val):
        return []
    return [t.strip() for t in re.split(r"[;,]", str(val))
            if t.strip() and t.strip().lower() != "nan"]


def build_promoter_seq(row, cols):
    parts = []
    for col in cols:
        val = row.get(col, "")
        if pd.notna(val) and str(val).strip() not in ("", "nan"):
            seq = str(val).strip().upper()
            if col == "UP":
                seq = seq[-19:]  # 只取末尾 19 bp
            parts.append(seq)
    return "".join(parts)


def trim_up(val):
    s = str(val).strip()
    if pd.notna(val) and s not in ("", "nan"):
        return s[-19:].upper()
    return ""


def count_hits(df_col, gene_set):
    """欄位中任一 token 命中 gene_set 即算該列命中。"""
    return int(df_col.apply(lambda v: any(t in gene_set for t in split_ids(v))).sum())


def pick_join_col(df, annotated):
    hp = count_hits(df["Locus_tag"],       annotated) if "Locus_tag"       in df.columns else 0
    ho = count_hits(df["Locus_tag (old)"], annotated) if "Locus_tag (old)" in df.columns else 0
    join = "Locus_tag" if hp >= ho else "Locus_tag (old)"
    print(f"  Hit counts -> Locus_tag: {hp} | Locus_tag (old): {ho}  =>  join: {join}")
    return join


POOLS = ["4cat", "other_annotated", "unannotated"]


def process_species(sp, df, gene_4cat, annotated, n_target, rng):
    print(f"\n{'='*60}\nSpecies: {sp}  (N={n_target})")
    join_col = pick_join_col(df, annotated)

    pools = {p: [] for p in POOLS}
    for _, row in df.iterrows():
        promoter = build_promoter_seq(row, PROMOTER_COLS)
        if not promoter:
            continue  # 無有效序列 -> 排除

        tokens = split_ids(row.get(join_col, ""))
        cats, is_annot = set(), False
        for t in tokens:
            if t in annotated:
                is_annot = True
            if t in gene_4cat:
                cats |= gene_4cat[t]

        pool = "4cat" if cats else ("other_annotated" if is_annot else "unannotated")
        pools[pool].append({
            "Species":         sp,
            "Locus_tag":       row.get("Locus_tag", ""),
            "Locus_tag (old)": row.get("Locus_tag (old)", ""),
            "Gene_name":       row.get("Gene_name", ""),
            "Pool":            pool,
            "Category":        ",".join(sorted(cats)),
            "multi_category":  len(cats) > 1,
            "TSS":             row.get("TSS", ""),
            "Direction":       row.get("Direction", ""),
            "UP":              trim_up(row.get("UP", "")),
            "Minus35":         row.get("Minus35", ""),
            "Spacer":          row.get("Spacer", ""),
            "Minus10":         row.get("Minus10", ""),
            "Dis":             row.get("Dis", ""),
            "Start":           row.get("Start", ""),
            "ITR":             row.get("ITR", ""),
            "Promoter_seq":    promoter,
        })

    # 分層挑選 A -> B -> C
    selected = []
    for tier in POOLS:
        if len(selected) >= n_target:
            break
        need, bucket = n_target - len(selected), pools[tier]
        selected.extend(bucket if len(bucket) <= need else rng.sample(bucket, need))

    result_df = pd.DataFrame(selected, columns=OUTPUT_COLS)

    avail   = {p: len(pools[p]) for p in POOLS}
    sel     = result_df["Pool"].value_counts().to_dict()
    cat_cnt = {c: int(result_df["Category"].apply(lambda x: c in str(x).split(",")).sum())
               for c in CATEGORIES}
    n_sel   = len(result_df)
    short   = max(0, n_target - n_sel)

    print(f"  Available -> A(4cat)={avail['4cat']} B(other)={avail['other_annotated']} "
          f"C(unannot)={avail['unannotated']}  total={sum(avail.values())}")
    print(f"  Selected={n_sel}/{n_target}" + (f"  *** SHORT by {short} ***" if short else "")
          + f"  | from A={sel.get('4cat',0)} B={sel.get('other_annotated',0)} "
          f"C={sel.get('unannotated',0)}")
    print("  4cat breakdown among selected: "
          + " | ".join(f"{c}={cat_cnt[c]}" for c in CATEGORIES))

    summary = {
        "Species": sp, "N_target": n_target, "Selected": n_sel, "Short": short,
        "sel_4cat": sel.get("4cat", 0),
        "sel_other": sel.get("other_annotated", 0),
        "sel_unannot": sel.get("unannotated", 0),
        **{c: cat_cnt[c] for c in CATEGORIES},
        "avail_4cat": avail["4cat"], "avail_other": avail["other_annotated"],
        "avail_unannot": avail["unannotated"],
    }
    return result_df, summary


results      = {}
summary_rows = []
for sp in SPECIES_LIST:
    rng = random.Random(SEED)  # 每株獨立、可重現
    res_df, summ = process_species(sp, sheets[sp], kegg_4cat[sp], kegg_annot[sp],
                                   N_TARGETS[sp], rng)
    results[sp] = res_df
    summary_rows.append(summ)

summary_df = pd.DataFrame(summary_rows)
print("\n" + "="*60 + "\nSummary:")
print(summary_df.to_string(index=False))


Species: Sy.el  (N=750)
  Hit counts -> Locus_tag: 1328 | Locus_tag (old): 0  =>  join: Locus_tag
  Available -> A(4cat)=306 B(other)=1022 C(unannot)=1101  total=2429
  Selected=750/750  | from A=306 B=444 C=0
  4cat breakdown among selected: TL=77 | NM=51 | AM=141 | SR=48

Species: My.sm  (N=750)
  Hit counts -> Locus_tag: 0 | Locus_tag (old): 1326  =>  join: Locus_tag (old)
  Available -> A(4cat)=334 B(other)=992 C(unannot)=1717  total=3043
  Selected=750/750  | from A=334 B=416 C=0
  4cat breakdown among selected: TL=104 | NM=53 | AM=154 | SR=41

Species: Ba.su  (N=750)
  Hit counts -> Locus_tag: 1 | Locus_tag (old): 336  =>  join: Locus_tag (old)
  Available -> A(4cat)=59 B(other)=277 C(unannot)=264  total=600
  Selected=600/750  *** SHORT by 150 ***  | from A=59 B=277 C=264
  4cat breakdown among selected: TL=17 | NM=11 | AM=18 | SR=14

Species: Ps.ae  (N=750)
  Hit counts -> Locus_tag: 0 | Locus_tag (old): 1304  =>  join: Locus_tag (old)
  Available -> A(4cat)=351 B(other)=953 C

## Step 4 — 寫出 Excel

- 每株一個 sheet（名稱 = 物種 acronym），只含被挑中的啟動子。
- `Summary` sheet：每株的 `N_target` / `Selected` / `Short`、各池被選數（`sel_4cat`/`sel_other`/`sel_unannot`）、四類細項，以及各池可用總數（`avail_*`）。

In [10]:
print(f"\nWriting {OUTPUT_FILE} ...")
with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    for sp in SPECIES_LIST:
        results[sp].to_excel(writer, sheet_name=sp, index=False)
    summary_df.to_excel(writer, sheet_name="Summary", index=False)
print("Done.")


Writing C:\project\Whole-model\MS2_Data_PyTorch\scripts\library_release\data\native\native_promoters_12species.xlsx ...
Done.


## Step 5: standardised output for the assembly step

Writes `outputs/02_native_promoters.csv`. Self-contained: re-reads the
Excel produced above, so it can run without redoing the selection.

In [11]:
# === Step 5: standardised native table (02_native_promoters.csv) ===
# Native promoters keep their own length (50-66 nt here): unlike the designed
# set we do not know where the real element boundaries sit, so nothing is
# padded or trimmed. BG5/BG3, RE sites and barcode are added later, in 06.
import pandas as pd

src = require(NATIVE_DIR / "native_promoters_12species.xlsx", "native selection output")
xl = pd.ExcelFile(src)
sheets = [s for s in xl.sheet_names if s != "Summary"]
print("species sheets:", len(sheets), sheets)

native = pd.concat([xl.parse(s, dtype=str) for s in sheets], ignore_index=True)
print("rows:", len(native))

RENAME = {
    "Promoter_seq": "promoter_sequence",
    "Locus_tag": "native_locus_tag",
    "Species": "organism",
    "Pool": "native_kegg_pool",
    "Category": "native_kegg_category",
}
missing = [c for c in RENAME if c not in native.columns]
if missing:
    raise KeyError(f"native sheet missing expected columns: {missing}")
native = native.rename(columns=RENAME)

# Known contamination: some Dis / Promoter_seq cells carry a literal '*'.
# standardize() flags these through alphabet_valid, so they are excluded from
# qc_pass instead of silently entering the library.
star = native["promoter_sequence"].astype(str).str.contains(r"\*", na=False)
print(f"rows containing '*': {int(star.sum())}")

out = standardize(native, source="native", candidate_id="NAT-")
out.to_csv(RELEASE_OUT / "02_native_promoters.csv", index=False)

print(f"\nwrote {RELEASE_OUT / '02_native_promoters.csv'}  {out.shape}")
print("promoter_length range:", int(out['promoter_length'].min()), "-", int(out['promoter_length'].max()))
print("qc_pass:", int(out["qc_pass"].sum()), "/", len(out))
print(out[STD_COLS].head(3).to_string(index=False))


species sheets: 12 ['Sy.el', 'My.sm', 'Ba.su', 'Ps.ae', 'Es.co', 'My.tu', 'St.au', 'Ba.th', 'He.py', 'Ag.tu', 'Bu.ce', 'Sa.ty']
rows: 5700
rows containing '*': 186

wrote C:\project\Whole-model\MS2_Data_PyTorch\scripts\library_release\outputs\02_native_promoters.csv  (5700, 22)
promoter_length range: 50 - 66
qc_pass: 5514 / 5700
candidate_id source                                                promoter_sequence  promoter_length  alphabet_valid  qc_pass
   NAT-00001 native CAAACCAGCGATAATTTCGTTGCCGACCGGAAAACGCTCTGGCAATTTGCAAGATTAGTCTTGC               64            True     True
   NAT-00002 native         CTTCAAGCTGCTTTCAGAGTCGAGTCGGTTGCAAGTGCTGTGCACCCTGAAGGAAG               56            True     True
   NAT-00003 native    GGTAAAGCCTCCTTTTCGGCAGATGACCTCTTGGCTAACCTTAAAGCCCTGCAGGAAACCA               61            True     True
